In [1]:
# Cài đặt thư viện kaggle để kéo data
!pip install -q kaggle

from google.colab import files
import os

print("Nhớ upload file kaggle.json tải từ mục Legacy API lúc nãy nha:")
uploaded = files.upload()

# Tạo folder ẩn và chuyển file json vào để hệ thống cấp quyền
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("[OK] Config xong API Kaggle rồi!")

Nhớ upload file kaggle.json tải từ mục Legacy API lúc nãy nha:


Saving kaggle.json to kaggle.json
[OK] Config xong API Kaggle rồi!


In [6]:
# Dùng lệnh API kéo thẳng bộ Olist về Colab cho lẹ, khỏi tốn dung lượng máy
print("Đang kéo cục data từ Kaggle về...")
!kaggle datasets download -d olistbr/brazilian-ecommerce

import zipfile
print("Đang giải nén file zip...")
with zipfile.ZipFile("brazilian-ecommerce.zip", 'r') as zip_ref:
    zip_ref.extractall("olist_data")

print("[XONG] Giải nén thành công!")

Đang kéo cục data từ Kaggle về...
Dataset URL: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
License(s): CC-BY-NC-SA-4.0
brazilian-ecommerce.zip: Skipping, found more recently modified local copy (use --force to force download)
Đang giải nén file zip...
[XONG] Giải nén thành công!


In [3]:
import pandas as pd

print("Bắt đầu đọc data vào Pandas...")
# Bộ data gốc có 9 bảng, đồ án này mình chỉ lấy 3 bảng cốt lõi để làm ETL thôi
df_orders = pd.read_csv("olist_data/olist_orders_dataset.csv")
df_items = pd.read_csv("olist_data/olist_order_items_dataset.csv")
df_products = pd.read_csv("olist_data/olist_products_dataset.csv")

# In ra xem thử kích thước từng bảng
print(f"Bảng Orders: {df_orders.shape[0]} dòng")
print(f"Bảng Items: {df_items.shape[0]} dòng")
print(f"Bảng Products: {df_products.shape[0]} dòng")

Bắt đầu đọc data vào Pandas...
Bảng Orders: 99441 dòng
Bảng Items: 112650 dòng
Bảng Products: 32951 dòng


In [7]:
print("Bắt đầu dọn dẹp và join data (Transform)...")

# 1. Chỉ lấy mấy đơn hàng giao thành công, đơn hủy hoặc đang giao thì bỏ qua
df_orders_clean = df_orders[df_orders['order_status'] == 'delivered'].copy()

# 2. Xóa mấy dòng bị thiếu ngày giao hàng (Xử lý null values)
df_orders_clean = df_orders_clean.dropna(subset=['order_delivered_customer_date'])

# 3. Join 3 bảng lại với nhau để tạo thành 1 bảng hoàn chỉnh (Merge)
# Nối Orders với Items thông qua khóa order_id
df_merged = pd.merge(df_orders_clean, df_items, on='order_id', how='inner')

# Nối tiếp với Products thông qua khóa product_id
df_final = pd.merge(df_merged, df_products, on='product_id', how='inner')

# 4. Giữ lại mấy cột cần thiết cho việc phân tích thôi để tối ưu RAM
cols = [
    'order_id', 'customer_id', 'order_status',
    'product_id', 'product_category_name', 'price'
]
df_final = df_final[cols]

print(f"[XONG] Data đã sạch sẽ Tổng cộng còn {df_final.shape[0]} dòng.")
# In thử 3 dòng đầu ra xem form data chuẩn chưa
display(df_final.head(3))

Bắt đầu dọn dẹp và join data (Transform)...
[XONG] Data đã sạch sẽ Tổng cộng còn 110189 dòng.


,order_id,customer_id,order_status,product_id,product_category_name,price
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,87285b34884572647811a353c7ac498a,utilidades_domesticas,29.99
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,595fac2a385ac33a80bd5114aec74eb8,perfumaria,118.70
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,aa4383b373c6aca5d8797843e5594415,automotivo,159.90


In [8]:
import sqlite3

# Tạo file database SQLite ảo ngay trên môi trường Colab
db_path = "ecommerce_cleaned.db"
conn = sqlite3.connect(db_path)
print(f"Đã tạo DB: {db_path}")

# Đổ cục data từ Pandas DataFrame thẳng vào bảng SQL
# index=False để bỏ cột số thứ tự, if_exists='replace' để lỡ chạy lại thì nó ghi đè file cũ
df_final.to_sql('cleaned_sales_data', conn, index=False, if_exists='replace')
print(" Đã load data vào DB xong!")

# Test hệ thống bằng 1 câu query SQL thực tế
# (Tìm 5 ngành hàng mang lại doanh thu cao nhất)
test_query = """
SELECT product_category_name, COUNT(order_id) as total_orders, SUM(price) as total_revenue
FROM cleaned_sales_data
WHERE product_category_name IS NOT NULL
GROUP BY product_category_name
ORDER BY total_revenue DESC
LIMIT 5;
"""

print("\n--- TEST KẾT QUẢ BẰNG TRUY VẤN SQL ---")
test_df = pd.read_sql_query(test_query, conn)
display(test_df)

# Xong việc nhớ đóng connection lại
conn.close()

Đã tạo DB: ecommerce_cleaned.db
 Đã load data vào DB xong!

--- TEST KẾT QUẢ BẰNG TRUY VẤN SQL ---


,product_category_name,total_orders,total_revenue
0,beleza_saude,9465,1233131.72
1,relogios_presentes,5857,1165898.98
2,cama_mesa_banho,10953,1023434.76
3,esporte_lazer,8430,954673.55
4,informatica_acessorios,7643,888613.62
